# Ekstrakcija ključnih tačaka iz videa (ASL Citizen)

Ovaj notebook se pokreće JEDNOM. Pretvara video snimke znakova u nizove
ključnih tačaka koje koristi model.

Tok:
1. Preuzima se ASL Citizen dataset (Microsoft Research) i raspakuje se podskup
   od 25 izabranih znakova (973 videa: 497 train, 96 val, 380 test).
2. Svaki video se propušta kroz MediaPipe (HandLandmarker + PoseLandmarker),
   frejm po frejm. Iz svakog frejma se izvlači:
   - leva šaka: 21 tačka × 3 (x,y,z) = 63
   - desna šaka: 21 tačka × 3 = 63
   - poza (ramena, laktovi, zglobovi): 12 tačaka × 3 = 36
   Ukupno 162 vrednosti po frejmu. Lice se izostavlja jer ne nosi informaciju
   za izolovane znakove.
3. Rezultat svakog videa (niz [broj_frejmova × 162]) snima se kao .npy fajl
   na Google Drive. Ti .npy fajlovi su ulaz za notebook sa obukom.

Napomena: ekstrakcija je spora (MediaPipe radi frejm po frejm na CPU) i radi
se samo jednom. Dalji rad (priprema, modeli, obuka) koristi gotove .npy tačke.

## Ekstrakcija: video → ključne tačke

Za svaki video, MediaPipe detektuje položaj šaka i tela u svakom frejmu i
vraća koordinate tačaka. Tačke koje nisu detektovane (npr. šaka van kadra)
popunjavaju se nulama. Sve sekvence se snimaju kao .npy na Drive, uz preskakanje
već obrađenih (da se ekstrakcija može nastaviti ako se sesija prekine).

In [ ]:
import os
print(os.path.exists("/content/drive/MyDrive"))

True


In [ ]:
!wget -q --show-progress \
  https://download.microsoft.com/download/b/8/8/b88c0bae-e6c1-43e1-8726-98cf5af36ca4/ASL_Citizen.zip \
  -O /content/ASL_Citizen.zip
print("Skinuto.")

/content/ASL_Citize 100%[===================>]  42.77G   109MB/s    in 9m 3s   
Skinuto.


In [ ]:
import zipfile
z = zipfile.ZipFile("/content/ASL_Citizen.zip")
names = z.namelist()
print("Ukupno stavki:", len(names))
for n in names[:20]:
    print(n)
csvs = [n for n in names if n.endswith(".csv")]
print("\nCSV fajlovi:")
for c in csvs:
    print(" ", c)

Ukupno stavki: 83406
ASL_Citizen/
ASL_Citizen/splits/
ASL_Citizen/splits/val.csv
ASL_Citizen/splits/train.csv
ASL_Citizen/splits/test.csv
ASL_Citizen/videos/
ASL_Citizen/videos/5086723486875651-RUN.mp4
ASL_Citizen/videos/2233415645150214-SWEET.mp4
ASL_Citizen/videos/09412352698319548-POSSIBLE.mp4
ASL_Citizen/videos/6184880487426632-PAINT.mp4
ASL_Citizen/videos/6682045893304822-RACE.mp4
ASL_Citizen/videos/22425009794174322-SCULPT.mp4
ASL_Citizen/videos/8563900568328626-SCAR MEMORY.mp4
ASL_Citizen/videos/6843553981288921-EARTHQUAKE.mp4
ASL_Citizen/videos/37687322393253075-REPLACE.mp4
ASL_Citizen/videos/8136380896447564-WRISTWATCH 3.mp4
ASL_Citizen/videos/023749434963018734-WHISTLE.mp4
ASL_Citizen/videos/639898672430512-PARTY 2.mp4
ASL_Citizen/videos/2743155783934501-ELEGANCE.mp4
ASL_Citizen/videos/11137281856386338-MENTION.mp4

CSV fajlovi:
  ASL_Citizen/splits/val.csv
  ASL_Citizen/splits/train.csv
  ASL_Citizen/splits/test.csv


In [ ]:
import zipfile, pandas as pd, os

WORK = "/content/drive/MyDrive/asl_citizen"
os.makedirs(f"{WORK}/splits", exist_ok=True)

z = zipfile.ZipFile("/content/ASL_Citizen.zip")
for split in ["train.csv", "val.csv", "test.csv"]:
    src = f"ASL_Citizen/splits/{split}"
    with z.open(src) as f, open(f"{WORK}/splits/{split}", "wb") as out:
        out.write(f.read())
print("CSV-ovi sačuvani na Drive.")

train = pd.read_csv(f"{WORK}/splits/train.csv")
print("\nKolone:", list(train.columns))
print("Redova u train:", len(train))
print(train.head())

CSV-ovi sačuvani na Drive.

Kolone: ['Participant ID', 'Video file', 'Gloss', 'ASL-LEX Code']
Redova u train: 40154
  Participant ID                        Video file       Gloss ASL-LEX Code
0             P1       15890366051589533-APPLE.mp4       APPLE     A_03_054
1             P1  35618482303951104-IMPOSSIBLE.mp4  IMPOSSIBLE     B_01_032
2             P1         6958143575951994-PARK.mp4        PARK     E_03_028
3             P1     8006032738002744-SOCCER 2.mp4     SOCCER2     F_03_032
4             P1       37542279833186454-STINK.mp4       STINK     H_01_064


In [ ]:
gloss_col = [c for c in train.columns if 'gloss' in c.lower()][0]
print("Kolona sa znakom:", gloss_col)
print("Broj različitih znakova:", train[gloss_col].nunique())

vc = train[gloss_col].value_counts()
print("Primera po znaku — min:", vc.min(), "max:", vc.max(), "mean:", round(vc.mean(),1))
print("\nPrimer nekoliko znakova:")
print(vc.head(15))

Kolona sa znakom: Gloss
Broj različitih znakova: 2731
Primera po znaku — min: 9 max: 24 mean: 14.7

Primer nekoliko znakova:
Gloss
DOG1             24
HURDLE/TRIP1     22
DEMAND1          21
BITE1            21
DARK1            21
BREAKFAST1       21
MECHANIC1        20
PARTY1           20
ROCKINGCHAIR1    20
DEAF1            20
WHATFOR1         20
DECIDE1          20
FINE1            19
NOON1            19
HALLOWEEN1       19
Name: count, dtype: int64


In [ ]:
import pandas as pd

WORK = "/content/drive/MyDrive/asl_citizen"
train = pd.read_csv(f"{WORK}/splits/train.csv")
val   = pd.read_csv(f"{WORK}/splits/val.csv")
test  = pd.read_csv(f"{WORK}/splits/test.csv")

# Znakovi koji postoje u sva tri splita
common = set(train.Gloss) & set(val.Gloss) & set(test.Gloss)
print("Znakova u sva tri splita:", len(common))

# Medju njima, biramo onih 25 sa najvise primera u trainu
counts = train[train.Gloss.isin(common)].Gloss.value_counts()
SELECTED = counts.head(25).index.tolist()

print("\nIzabrani znakovi (25):")
for s in SELECTED:
    n_tr = (train.Gloss == s).sum()
    n_va = (val.Gloss == s).sum()
    n_te = (test.Gloss == s).sum()
    print(f"  {s:18s} train={n_tr:2d}  val={n_va}  test={n_te}")

# Sačuvaj listu da je koristimo kasnije
import json
with open(f"{WORK}/selected_signs.json", "w") as f:
    json.dump(SELECTED, f)
print("\nLista sačuvana u selected_signs.json")

Znakova u sva tri splita: 2731

Izabrani znakovi (25):
  DOG1               train=24  val=4  test=17
  HURDLE/TRIP1       train=22  val=3  test=13
  DEMAND1            train=21  val=4  test=14
  BITE1              train=21  val=4  test=14
  DARK1              train=21  val=3  test=15
  BREAKFAST1         train=21  val=3  test=15
  MECHANIC1          train=20  val=3  test=16
  PARTY1             train=20  val=4  test=15
  ROCKINGCHAIR1      train=20  val=4  test=15
  DEAF1              train=20  val=4  test=15
  WHATFOR1           train=20  val=5  test=18
  DECIDE1            train=20  val=3  test=15
  FINE1              train=19  val=4  test=16
  NOON1              train=19  val=4  test=15
  HALLOWEEN1         train=19  val=4  test=14
  DEVELOP1           train=19  val=3  test=14
  LUNCH1             train=19  val=5  test=14
  EDIT1              train=19  val=4  test=15
  SHAVE1             train=19  val=5  test=15
  BACKPACK1          train=19  val=3  test=15
  BEE1               trai

In [ ]:
import zipfile, pandas as pd, json, os

WORK = "/content/drive/MyDrive/asl_citizen"
with open(f"{WORK}/selected_signs.json") as f:
    SELECTED = json.load(f)

train = pd.read_csv(f"{WORK}/splits/train.csv")
val   = pd.read_csv(f"{WORK}/splits/val.csv")
test  = pd.read_csv(f"{WORK}/splits/test.csv")

# Skupi imena svih video fajlova koji pripadaju izabranim znakovima
wanted = {}  # split -> lista video fajlova
for name, df in [("train", train), ("val", val), ("test", test)]:
    files = df[df.Gloss.isin(SELECTED)]["Video file"].tolist()
    wanted[name] = files
    print(f"{name}: {len(files)} videa")

total = sum(len(v) for v in wanted.values())
print("Ukupno videa za raspakivanje:", total)

# Raspakuj ih na Drive, organizovano po splitu
z = zipfile.ZipFile("/content/ASL_Citizen.zip")
for split, files in wanted.items():
    out_dir = f"{WORK}/videos/{split}"
    os.makedirs(out_dir, exist_ok=True)
    for vf in files:
        src = f"ASL_Citizen/videos/{vf}"
        try:
            with z.open(src) as fsrc, open(f"{out_dir}/{vf}", "wb") as fout:
                fout.write(fsrc.read())
        except KeyError:
            print("  Nedostaje u zipu:", vf)
    print(f"{split}: raspakovano u {out_dir}")

print("Gotovo.")

train: 497 videa
val: 96 videa
test: 380 videa
Ukupno videa za raspakivanje: 973
train: raspakovano u /content/drive/MyDrive/asl_citizen/videos/train
val: raspakovano u /content/drive/MyDrive/asl_citizen/videos/val
test: raspakovano u /content/drive/MyDrive/asl_citizen/videos/test
Gotovo.


In [ ]:
!pip uninstall -y mediapipe protobuf -q
!pip install -q --upgrade mediapipe
print("Instalirano. OBAVEZNO: Runtime -> Restart session, pa kreni od Ćelije 8b.")

Instalirano. OBAVEZNO: Runtime -> Restart session, pa kreni od Ćelije 8b.


In [ ]:
import mediapipe as mp
print("Verzija:", mp.__version__)
from mediapipe.tasks import python
from mediapipe.tasks.python import vision
print("Tasks API dostupan:", vision is not None)

Verzija: 0.10.35
Tasks API dostupan: True


In [ ]:
!wget -q -O /content/hand_landmarker.task \
  https://storage.googleapis.com/mediapipe-models/hand_landmarker/hand_landmarker/float16/1/hand_landmarker.task
!wget -q -O /content/pose_landmarker.task \
  https://storage.googleapis.com/mediapipe-models/pose_landmarker/pose_landmarker_lite/float16/1/pose_landmarker_lite.task
print("Modeli skinuti.")

Modeli skinuti.


In [ ]:
import numpy as np

N_HAND = 21
# Gornji deo tela: ramena, laktovi, zglobovi (poza ima 33 tačke; uzimamo 11-22)
POSE_IDX = [11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22]
N_POSE = len(POSE_IDX)

# 2 šake * 21 * 3  +  12 tačaka poze * 3
FEATURES_PER_FRAME = N_HAND*3*2 + N_POSE*3   # 126 + 36 = 162
print("Karakteristika po frejmu:", FEATURES_PER_FRAME)

Karakteristika po frejmu: 162


In [ ]:
import cv2
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision

BaseOptions = python.BaseOptions
VisionRunningMode = vision.RunningMode

def make_landmarkers():
    hand_opts = vision.HandLandmarkerOptions(
        base_options=BaseOptions(model_asset_path="/content/hand_landmarker.task"),
        running_mode=VisionRunningMode.VIDEO,
        num_hands=2,
        min_hand_detection_confidence=0.5,
        min_tracking_confidence=0.5,
    )
    pose_opts = vision.PoseLandmarkerOptions(
        base_options=BaseOptions(model_asset_path="/content/pose_landmarker.task"),
        running_mode=VisionRunningMode.VIDEO,
        num_poses=1,
        min_pose_detection_confidence=0.5,
        min_tracking_confidence=0.5,
    )
    hand = vision.HandLandmarker.create_from_options(hand_opts)
    pose = vision.PoseLandmarker.create_from_options(pose_opts)
    return hand, pose

def extract_keypoints_from_video(video_path):
    """Vrati [T, 162]: leva šaka(63) + desna šaka(63) + poza(36). Fali -> nule."""
    cap = cv2.VideoCapture(video_path)
    fps = cap.get(cv2.CAP_PROP_FPS) or 30
    hand_lm, pose_lm = make_landmarkers()
    frames = []
    idx = 0
    while cap.isOpened():
        ok, frame = cap.read()
        if not ok:
            break
        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)
        ts = int(idx * 1000 / fps)

        # ŠAKE
        left = np.zeros(N_HAND*3)
        right = np.zeros(N_HAND*3)
        hres = hand_lm.detect_for_video(mp_image, ts)
        if hres.hand_landmarks:
            for hand_pts, handed in zip(hres.hand_landmarks, hres.handedness):
                pts = np.array([[p.x, p.y, p.z] for p in hand_pts]).flatten()
                if handed[0].category_name == "Left":
                    left = pts
                else:
                    right = pts

        # POZA
        pose = np.zeros(N_POSE*3)
        pres = pose_lm.detect_for_video(mp_image, ts)
        if pres.pose_landmarks:
            lms = pres.pose_landmarks[0]
            pose = np.array([[lms[i].x, lms[i].y, lms[i].z] for i in POSE_IDX]).flatten()

        frames.append(np.concatenate([left, right, pose]))
        idx += 1
    cap.release()
    hand_lm.close()
    pose_lm.close()
    return np.array(frames, dtype=np.float32)

# Test na jednom videu
import glob
test_video = glob.glob("/content/drive/MyDrive/asl_citizen/videos/train/*.mp4")[0]
kp = extract_keypoints_from_video(test_video)
print("Test video:", test_video.split("/")[-1])
print("Oblik izlaza [frejmova, karakteristika]:", kp.shape)

Test video: 4520498201410337-BACKPACK.mp4
Oblik izlaza [frejmova, karakteristika]: (76, 162)


In [ ]:
import glob, os, time
import numpy as np

WORK = "/content/drive/MyDrive/asl_citizen"

for split in ["train", "val", "test"]:
    in_dir = f"{WORK}/videos/{split}"
    out_dir = f"{WORK}/keypoints/{split}"
    os.makedirs(out_dir, exist_ok=True)

    videos = sorted(glob.glob(f"{in_dir}/*.mp4"))
    print(f"\n=== {split}: {len(videos)} videa ===")
    t0 = time.time()

    for i, vp in enumerate(videos):
        name = os.path.splitext(os.path.basename(vp))[0]
        out_path = f"{out_dir}/{name}.npy"

        # Preskoči ako je već urađeno (da možeš da nastaviš ako sesija pukne)
        if os.path.exists(out_path):
            continue

        try:
            kp = extract_keypoints_from_video(vp)
            np.save(out_path, kp)
        except Exception as e:
            print(f"  GREŠKA na {name}: {e}")

        if (i+1) % 50 == 0:
            elapsed = time.time() - t0
            print(f"  {i+1}/{len(videos)}  ({elapsed:.0f}s)")

    print(f"{split} gotov za {time.time()-t0:.0f}s")

print("\nSVE GOTOVO.")


=== train: 497 videa ===
  250/497  (49s)
  300/497  (294s)
  350/497  (513s)
  400/497  (749s)
  450/497  (969s)
train gotov za 1222s

=== val: 96 videa ===
  50/96  (276s)
val gotov za 550s

=== test: 380 videa ===
  50/380  (236s)
  100/380  (422s)
  150/380  (636s)
  200/380  (842s)
  250/380  (1057s)
  300/380  (1287s)
  350/380  (1531s)
test gotov za 1654s

SVE GOTOVO.


In [ ]:
import os, glob, shutil, json
WORK = "/content/drive/MyDrive/asl_citizen"
# 1. Provera da je ekstrakcija stvarno gotova
for split in ["train", "val", "test"]:
    n = len(glob.glob(f"{WORK}/keypoints/{split}/*.npy"))
    print(f"keypoints/{split}: {n} .npy")
# 2. Spakuj sve što treba za dalje u jedan folder
bundle = "/content/asl_bundle"
os.makedirs(bundle, exist_ok=True)
shutil.copytree(f"{WORK}/keypoints", f"{bundle}/keypoints", dirs_exist_ok=True)
shutil.copytree(f"{WORK}/splits", f"{bundle}/splits", dirs_exist_ok=True)
shutil.copy(f"{WORK}/selected_signs.json", f"{bundle}/selected_signs.json")
# 3. Napravi jedan zip i sačuvaj ga na Drive
shutil.make_archive(f"{WORK}/asl_keypoints_bundle", "zip", bundle)
size = os.path.getsize(f"{WORK}/asl_keypoints_bundle.zip")/1e6
print(f"\nSačuvano: {WORK}/asl_keypoints_bundle.zip  ({size:.0f} MB)")

keypoints/train: 497 .npy
keypoints/val: 96 .npy
keypoints/test: 380 .npy

Sačuvano: /content/drive/MyDrive/asl_citizen/asl_keypoints_bundle.zip  (23 MB)


In [ ]:
import os, glob, json
import numpy as np
import pandas as pd

WORK = "/content/drive/MyDrive/asl_citizen"

with open(f"{WORK}/selected_signs.json") as f:
    SELECTED = sorted(json.load(f))
sign_to_idx = {s: i for i, s in enumerate(SELECTED)}
idx_to_sign = {i: s for s, i in sign_to_idx.items()}
print("Broj klasa:", len(SELECTED))

# Napravi mapu: ime_fajla (bez .npy) -> Gloss, iz CSV-ova
file_to_gloss = {}
for split in ["train", "val", "test"]:
    df = pd.read_csv(f"{WORK}/splits/{split}.csv")
    for _, row in df.iterrows():
        fname = os.path.splitext(row["Video file"])[0]   # bez .mp4
        file_to_gloss[fname] = row["Gloss"]

def load_split(split):
    X, y, lengths = [], [], []
    for path in sorted(glob.glob(f"{WORK}/keypoints/{split}/*.npy")):
        name = os.path.basename(path)[:-4]          # ime bez .npy
        gloss = file_to_gloss.get(name)             # tačan Gloss iz CSV-a
        if gloss is None or gloss not in sign_to_idx:
            continue
        seq = np.load(path)
        if len(seq) == 0:
            continue
        X.append(seq)
        y.append(sign_to_idx[gloss])
        lengths.append(len(seq))
    return X, np.array(y), lengths

for split in ["train", "val", "test"]:
    X, y, L = load_split(split)
    if L:
        print(f"{split}: {len(X)} sekvenci, dužine min={min(L)} max={max(L)} med={int(np.median(L))}")
    else:
        print(f"{split}: 0 sekvenci — i dalje nešto ne valja")

Broj klasa: 25
train: 497 sekvenci, dužine min=25 max=505 med=75
val: 96 sekvenci, dužine min=37 max=206 med=71
test: 380 sekvenci, dužine min=10 max=234 med=71


In [ ]:
MAX_LEN = 64   # fiksna dužina sekvence (oko medijane)

def pad_or_truncate(seq, max_len=MAX_LEN):
    """Svede sekvencu na tačno max_len frejmova.
       Duže -> ravnomerno uzorkuje max_len frejmova.
       Kraće -> dopuni nulama na kraju.
       Vrati (sekvenca[max_len, F], maska[max_len])."""
    T = len(seq)
    F = seq.shape[1]
    if T >= max_len:
        # ravnomerno izaberi max_len indeksa kroz celu sekvencu
        idx = np.linspace(0, T-1, max_len).astype(int)
        out = seq[idx]
        mask = np.ones(max_len, dtype=np.float32)
    else:
        out = np.zeros((max_len, F), dtype=np.float32)
        out[:T] = seq
        mask = np.zeros(max_len, dtype=np.float32)
        mask[:T] = 1.0
    return out.astype(np.float32), mask

def build_split(split):
    X, y, _ = load_split(split)
    Xp = np.zeros((len(X), MAX_LEN, X[0].shape[1]), dtype=np.float32)
    M  = np.zeros((len(X), MAX_LEN), dtype=np.float32)
    for i, seq in enumerate(X):
        Xp[i], M[i] = pad_or_truncate(seq)
    return Xp, np.array(y), M

X_train, y_train, m_train = build_split("train")
X_val,   y_val,   m_val   = build_split("val")
X_test,  y_test,  m_test  = build_split("test")

print("Train:", X_train.shape, "labele:", y_train.shape)
print("Val:  ", X_val.shape)
print("Test: ", X_test.shape)
print("Primer maske (prva sekvenca, koliko pravih frejmova):", int(m_train[0].sum()))

Train: (497, 64, 162) labele: (497,)
Val:   (96, 64, 162)
Test:  (380, 64, 162)
Primer maske (prva sekvenca, koliko pravih frejmova): 64


In [ ]:
# Indeksi unutar vektora od 162:
# leva šaka: 0..62 (21 tačka), desna: 63..125, poza: 126..161 (12 tačaka)
# U pozi su prve dve tačke ramena (idx 11,12 u originalu -> ovde prvih 6 brojeva)

def normalize(X, mask):
    """Centriraj svaki frejm oko sredine ramena i skaliraj po širini ramena."""
    Xn = X.copy()
    N, T, F = X.shape
    for i in range(N):
        for t in range(T):
            if mask[i, t] == 0:
                continue  # prazan (dopunjeni) frejm preskoči
            frame = X[i, t].reshape(-1, 3)  # 54 tačke x 3
            # ramena su prve dve tačke poze: indeks poze počinje na 126/3 = 42
            l_sh = frame[42]   # levo rame
            r_sh = frame[43]   # desno rame
            center = (l_sh + r_sh) / 2
            scale = np.linalg.norm(l_sh - r_sh) + 1e-6  # širina ramena
            # centriraj i skaliraj samo tačke koje nisu nula (viđene)
            nonzero = np.any(frame != 0, axis=1)
            frame[nonzero] = (frame[nonzero] - center) / scale
            Xn[i, t] = frame.flatten()
    return Xn

X_train_n = normalize(X_train, m_train)
X_val_n   = normalize(X_val,   m_val)
X_test_n  = normalize(X_test,  m_test)
print("Normalizovano. Primer raspona vrednosti:")
print("  pre :", round(float(X_train[0][0].min()),2), "do", round(float(X_train[0][0].max()),2))
print("  posle:", round(float(X_train_n[0][0].min()),2), "do", round(float(X_train_n[0][0].max()),2))

Normalizovano. Primer raspona vrednosti:
  pre : -1.96 do 2.01
  posle: -1.96 do 2.01


In [ ]:
# Uzmi jedan frejm i pogledaj šta je na poziciji ramena
frame = X_train[0][0].reshape(-1, 3)
print("Ukupno tačaka u frejmu:", frame.shape[0])   # treba 54
print("Tačka 42 (levo rame?):", frame[42])
print("Tačka 43 (desno rame?):", frame[43])
print("\nKoliko tačaka je nula (neviđeno) u ovom frejmu:")
print("  nula tačaka:", int(np.sum(np.all(frame==0, axis=1))), "od 54")

# Da li poza uopšte postoji u ovom frejmu? Poza je zadnjih 36 brojeva = 12 tačaka
poza = X_train[0][0][126:].reshape(-1,3)
print("\nPoza (12 tačaka), prve 3:")
print(poza[:3])

Ukupno tačaka u frejmu: 54
Tačka 42 (levo rame?): [ 0.44644046 -0.02494104  0.2237581 ]
Tačka 43 (desno rame?): [-0.44644052  0.02494104 -0.2237581 ]

Koliko tačaka je nula (neviđeno) u ovom frejmu:
  nula tačaka: 42 od 54

Poza (12 tačaka), prve 3:
[[ 0.44644046 -0.02494104  0.2237581 ]
 [-0.44644052  0.02494104 -0.2237581 ]
 [ 0.5296957   0.7393709   0.41590694]]


In [ ]:
print("Ramena u NORMALIZOVANOM (X_train_n):")
fn = X_train_n[0][0].reshape(-1,3)
print("  levo:", fn[42], "\n  desno:", fn[43])
print("  sredina ramena (treba ~0):", (fn[42]+fn[43])/2)

print("\nRamena u ORIGINALNOM (X_train):")
fo = X_train[0][0].reshape(-1,3)
print("  levo:", fo[42], "\n  desno:", fo[43])

Ramena u NORMALIZOVANOM (X_train_n):
  levo: [ 0.44644046 -0.02494104  0.2237581 ] 
  desno: [-0.44644052  0.02494104 -0.2237581 ]
  sredina ramena (treba ~0): [-2.9802322e-08  0.0000000e+00  0.0000000e+00]

Ramena u ORIGINALNOM (X_train):
  levo: [ 0.44644046 -0.02494104  0.2237581 ] 
  desno: [-0.44644052  0.02494104 -0.2237581 ]


In [ ]:
# Koliki procenat frejmova ima bar jednu šaku detektovanu?
def hand_coverage(X, mask):
    total, with_hand = 0, 0
    for i in range(len(X)):
        for t in range(X.shape[1]):
            if mask[i,t]==0: continue
            total += 1
            hands = X[i,t][:126]  # obe šake
            if np.any(hands != 0):
                with_hand += 1
    return with_hand/total

print("Frejmova sa bar jednom šakom (train):", round(hand_coverage(X_train, m_train)*100,1), "%")

Frejmova sa bar jednom šakom (train): 43.4 %


In [ ]:
# Za nekoliko primera, pogledaj koji frejmovi imaju šake (1) a koji ne (0)
for i in range(3):
    has_hand = []
    for t in range(X_train.shape[1]):
        if m_train[i,t]==0:
            has_hand.append("-")  # padding
        elif np.any(X_train[i,t][:126] != 0):
            has_hand.append("█")  # ima šaku
        else:
            has_hand.append("·")  # nema šaku
    print(f"Primer {i} ({idx_to_sign[y_train[i]]}):")
    print("  " + "".join(has_hand))

Primer 0 (EDIT1):
  ··········██████████████████████······███████···················
Primer 1 (WHATFOR1):
  ············████████████████████································
Primer 2 (WHATFOR1):
  ·················██████████████████████████████████████████-----


In [ ]:
def trim_to_hands(seq):
    """Iseče sekvencu na opseg [prvi frejm sa šakom .. poslednji frejm sa šakom].
       Ako nema nijedne šake, vrati original."""
    has_hand = np.any(seq[:, :126] != 0, axis=1)  # po frejmu: ima li šaku
    if not has_hand.any():
        return seq
    first = np.argmax(has_hand)              # prvi True
    last  = len(has_hand) - np.argmax(has_hand[::-1])  # poslednji True +1
    return seq[first:last]

def build_split_trimmed(split):
    X, y, _ = load_split(split)
    Xp = np.zeros((len(X), MAX_LEN, X[0].shape[1]), dtype=np.float32)
    M  = np.zeros((len(X), MAX_LEN), dtype=np.float32)
    for i, seq in enumerate(X):
        seq = trim_to_hands(seq)             # <-- isečemo prazne krajeve
        Xp[i], M[i] = pad_or_truncate(seq)
    return Xp, np.array(y), M

# Ponovo izgradi splitove sa trimovanjem
X_train, y_train, m_train = build_split_trimmed("train")
X_val,   y_val,   m_val   = build_split_trimmed("val")
X_test,  y_test,  m_test  = build_split_trimmed("test")

# Normalizuj ponovo
X_train_n = normalize(X_train, m_train)
X_val_n   = normalize(X_val,   m_val)
X_test_n  = normalize(X_test,  m_test)

# Proveri novu pokrivenost šaka
def hand_coverage(X, mask):
    total, with_hand = 0, 0
    for i in range(len(X)):
        for t in range(X.shape[1]):
            if mask[i,t]==0: continue
            total += 1
            if np.any(X[i,t][:126] != 0): with_hand += 1
    return with_hand/total

print("Train oblik:", X_train.shape)
print("Pokrivenost šaka posle trimovanja:", round(hand_coverage(X_train, m_train)*100,1), "%")

Train oblik: (497, 64, 162)
Pokrivenost šaka posle trimovanja: 91.9 %
